# Teste isolado — ARTEMIG (Notícias)

Fonte candidata: **ARTEMIG - Agência Reguladora de Serviços Públicos
Delegados de Minas Gerais**, setor Transporte. Já existia placeholder em
`controle_fontes` (`source_id='—'`, `status='Não iniciada'`,
`importancia_original='Média'`). Notebook **descartável** (Fase 1) — sem
dispatcher, sem `atualizar_status_fonte`, sem gravar nada em produção.

URL fornecida: `https://artemig.mg.gov.br/`.

## Nota operacional — certificado TLS incompleto

O servidor devolve uma cadeia de certificado incompleta ("unable to get
local issuer certificate" -- falta o intermediário) para clientes que
validam contra o bundle padrão do SO. `httpx`/`requests` no Databricks
costumam usar o bundle do `certifi`, que às vezes já cobre o
intermediário faltante -- testar sem `verify=False` primeiro; só usar
`verify=False` como último recurso, e documentar se for necessário.

In [ ]:
%pip install --quiet httpx beautifulsoup4 lxml
dbutils.library.restartPython()

In [ ]:
import httpx
from bs4 import BeautifulSoup

BASE = "https://artemig.mg.gov.br"
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
)
HEADERS = {"User-Agent": USER_AGENT, "Accept-Language": "pt-BR,pt;q=0.9"}

## Teste 1 — robots.txt e a home

WordPress + Yoast SEO. `robots.txt` só bloqueia busca interna e
`/wp-json/` -- não impede scraping de conteúdo. A home tem uma seção
"Últimas notícias" com botão "Ver todas as notícias" apontando para
`/category/noticias/`.

In [ ]:
with httpx.Client(headers=HEADERS, timeout=30, follow_redirects=True) as client:
    resp_robots = client.get(f"{BASE}/robots.txt")
    resp_home = client.get(f"{BASE}/")

print(f"/robots.txt -> HTTP {resp_robots.status_code}\n{resp_robots.text}\n")
print(f"/ -> HTTP {resp_home.status_code}, {len(resp_home.text)} chars")

soup_home = BeautifulSoup(resp_home.text, "lxml")
titulo_secao = soup_home.find(string=lambda t: t and "Últimas notícias" in t)
print(f"Seção 'Últimas notícias' encontrada na home: {titulo_secao is not None}")

## Teste 2 — `/category/noticias/` (linkado pela própria home) está vazio

**Achado**: a categoria "Notícias" -- o próprio link "Ver todas as
notícias" da home aponta pra cá -- devolve "Nenhum post encontrado".
Bug/lacuna de conteúdo do lado da ARTEMIG, não nosso.

In [ ]:
with httpx.Client(headers=HEADERS, timeout=30, follow_redirects=True) as client:
    resp_cat = client.get(f"{BASE}/category/noticias/")

print(f"HTTP {resp_cat.status_code}, {len(resp_cat.text)} chars")
print(f"'Nenhum post encontrado' presente: {'Nenhum post encontrado' in resp_cat.text}")

## Teste 3 — `/posts/` (índice padrão do WordPress) tem posts, mas são
## páginas institucionais, não notícias

`post-sitemap.xml` (Yoast) mostrava atividade recente (última mod.
13/08/2026), então existe algum fluxo de publicação ativo. O índice de
posts de fato (`/posts/`, não a categoria vazia) lista itens reais --
mas verificando os títulos das duas primeiras páginas (20 itens, cobrindo
~1 ano, agosto/2025 a junho/2026), **nenhum é notícia**: são todos
páginas institucionais publicadas como `post` do WordPress (Plano de
Comunicação, Editais, Acessibilidade, LGPD, cada trecho de rodovia
concedida, etc.) -- o mesmo conteúdo que já aparece no menu institucional
do site, sem nenhum evento/decisão/anúncio datado.

In [ ]:
import re

titulos_todos = []
with httpx.Client(headers=HEADERS, timeout=30, follow_redirects=True) as client:
    for pagina in [1, 2]:
        url = f"{BASE}/posts/" if pagina == 1 else f"{BASE}/posts/page/{pagina}/"
        resp = client.get(url)
        titulos = re.findall(r'<h2 class="entry-title">\s*<a href="([^"]*)">([^<]*)</a>', resp.text)
        titulos_todos.extend(titulos)

print(f"{len(titulos_todos)} posts encontrados em /posts/ (2 páginas):\n")
for url, titulo in titulos_todos:
    print(f"  {titulo.strip()}")

## Conclusão da Fase 1

Site é WordPress comum (Elementor), sem WAF, `robots.txt` permite
scraping de conteúdo -- tecnicamente encaixaria fácil no dispatcher
genérico `ingest-scraping` (mesmo padrão de Acende Brasil, ABEGÁS,
AESBE...). **Mas não há o que capturar agora**: a própria ARTEMIG não
tem nenhuma notícia publicada no momento -- widget da home vazio,
categoria "Notícias" vazia ("Nenhum post encontrado"), e o único fluxo
de publicação ativo (`post` do WordPress, atualizado até 13/08/2026) é
só usado para páginas institucionais, não para notícias/eventos.

**Avaliação para a Fase 2**: não dá pra escrever um `listar_artemig()`
de verdade sem ter nenhum item real pra validar contra -- não
implementar agora. Diferente de um bloqueio técnico (WAF/robots.txt):
aqui é ausência de conteúdo do lado da fonte. Como o site claramente
segue ativo (páginas institucionais atualizadas até junho/2026, sitemap
até agosto/2026), vale retestar periodicamente -- se a ARTEMIG voltar a
publicar notícias reais (em `/category/noticias/` ou em qualquer URL
nova), a integração é direta (WordPress puro, sem obstáculo técnico).